# Reels Renderer — финальный самодостаточный ноутбук
Ассеты вшиты внутрь. Запускайте ячейки по порядку (▶). Runtime: T4 GPU.


In [ ]:
import os, urllib.request
ST = '/content/ST_MAIN'
if not os.path.isfile(os.path.join(ST, 'inference.py')):
    !git clone -q https://github.com/OpenTalker/SadTalker.git /content/ST_MAIN
os.chdir(ST)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('gfpgan/weights', exist_ok=True)
!apt-get -qq install -y ffmpeg
!pip uninstall -q -y torchaudio 2>/dev/null
!pip install -q torch==2.5.1 torchvision==0.20.1 kornia==0.7.3 gfpgan facexlib basicsr librosa==0.10.2 safetensors yacs pydub
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py
!sed -i "s/trans_params = np.array(\[w0, h0, s, t\[0\], t\[1\]\])/t = np.asarray(t).reshape(-1); s = float(np.asarray(s).reshape(-1)[0]); trans_params = np.array([w0, h0, s, float(t[0]), float(t[1])])/" /content/ST_MAIN/src/face3d/util/preprocess.py
!sed -i 's/np.array(\[float(item) for item in np.hsplit(trans_params, 5)\])/np.asarray(trans_params, dtype=np.float64).reshape(-1)/' /content/ST_MAIN/src/face3d/util/preprocess.py
D = [
 ('https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_256.safetensors','checkpoints/SadTalker_V0.0.2_256.safetensors'),
 ('https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_512.safetensors','checkpoints/SadTalker_V0.0.2_512.safetensors'),
 ('https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00109-model.pth.tar','checkpoints/mapping_00109-model.pth.tar'),
 ('https://github.com/Winfredy/SadTalker/releases/download/v0.0.2/wav2lip.pth','checkpoints/wav2lip.pth'),
 ('https://github.com/Winfredy/SadTalker/releases/download/v0.0.2/shape_predictor_68_face_landmarks.dat','checkpoints/shape_predictor_68_face_landmarks.dat'),
 ('https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth','gfpgan/weights/alignment_WFLW_4HG.pth'),
 ('https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth','gfpgan/weights/detection_Resnet50_Final.pth'),
 ('https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth','gfpgan/weights/parsing_parsenet.pth'),
 ('https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth','gfpgan/weights/GFPGANv1.4.pth'),
]
for u, f in D:
    if not os.path.isfile(f) or os.path.getsize(f) < 100000:
        print('качаю', f); urllib.request.urlretrieve(u, f)
if not os.path.isdir('checkpoints/BFM_Fitting'):
    urllib.request.urlretrieve('https://github.com/Winfredy/SadTalker/releases/download/v0.0.2/BFM_Fitting.zip','checkpoints/BFM_Fitting.zip')
    !cd checkpoints && unzip -oq BFM_Fitting.zip
print('ЯЧЕЙКА 1 ГОТОВА, cwd =', os.getcwd())


In [ ]:
import base64, os
os.makedirs('examples', exist_ok=True)
open('examples/avatar.png','wb').write(base64.b64decode('...'))
open('examples/voice5.wav','wb').write(base64.b64decode('...'))
print('АССЕТЫ:', os.path.getsize('examples/avatar.png'), os.path.getsize('examples/voice5.wav'))


In [ ]:
import numpy as np, sys, types
try:
    np.VisibleDeprecationWarning = np.exceptions.VisibleDeprecationWarning
except Exception:
    pass
for _a, _v in [('float', float), ('int', int), ('complex', complex)]:
    setattr(np, _a, _v)
import torchvision.transforms.functional as _F
_m = types.ModuleType('torchvision.transforms.functional_tensor')
_m.rgb_to_grayscale = _F.rgb_to_grayscale
sys.modules.setdefault('torchvision.transforms.functional_tensor', _m)
import runpy
sys.argv = ['inference', '--driven_audio', 'examples/voice5.wav', '--source_image', 'examples/avatar.png',
            '--checkpoint_dir', 'checkpoints', '--result_dir', 'results', '--preprocess', 'full',
            '--batch_size', '8', '--size', '512']
runpy.run_path('inference' + '.py', run_name='__main__')
print('РЕНДЕР ГОТОВ — запускайте ячейку 4')


In [ ]:
import glob, os, cv2, numpy as np, torch
from gfpgan import GFPGANer
raw = sorted(glob.glob('results/**/*.mp4', recursive=True))[-1]
vd = '/content/fr'; os.makedirs(vd, exist_ok=True)
for q in glob.glob(vd+'/*.png'): os.remove(q)
!ffmpeg -y -loglevel error -i {raw} /content/fr/f_%04d.png
restorer = GFPGANer(model_path='gfpgan/weights/GFPGANv1.4.pth', upscale=1, arch='clean', channel_multiplier=2)
orig = cv2.imread('examples/avatar.png')
FY0, FY1, FX0, FX1 = 150, 980, 140, 630
yy, xx = np.mgrid[0:1376, 0:768].astype(np.float32)
d = np.sqrt(((yy-560)/440.0)**2 + ((xx-385)/255.0)**2)
mask = np.clip((1.15-d)/0.35*0.5, 0, 1)[..., None]
mask = mask[FY0:FY1, FX0:FX1]
for p in sorted(glob.glob(vd+'/f_*.png')):
    img = cv2.resize(cv2.imread(p), (768,1376), interpolation=cv2.INTER_LANCZOS4)
    small = cv2.resize(img[FY0:FY1, FX0:FX1], (512,512))
    _, _, enh = restorer.enhance(small, has_aligned=False, only_center_face=True, paste_back=True)
    src = cv2.resize(orig[FY0:FY1, FX0:FX1], (512,512))
    det = src.astype(np.float32) - cv2.GaussianBlur(src,(0,0),2.0).astype(np.float32)
    enh = np.clip(enh.astype(np.float32)+det*0.6,0,255).astype(np.uint8)
    enh = cv2.resize(enh,(FX1-FX0,FY1-FY0))
    img[FY0:FY1,FX0:FX1] = (img[FY0:FY1,FX0:FX1].astype(np.float32)*(1-mask)+enh*mask).astype(np.uint8)
    b = cv2.GaussianBlur(img,(0,0),1.1); img = cv2.addWeighted(img,1.25,b,-0.25,0)
    cv2.imwrite(p,img)
!ffmpeg -y -loglevel error -framerate 25 -i /content/fr/f_%04d.png -i examples/voice5.wav -map 0:v -map 1:a -c:v libx264 -crf 18 -pix_fmt yuv420p -movflags +faststart -c:a aac -b:a 128k -shortest /content/result.mp4
def _save(path, name):
    try:
        import google.colab.files as _g
        _g.download(path)
    except BaseException:
        import shutil, os
        os.makedirs('/kaggle/working', exist_ok=True)
        shutil.copy(path, '/kaggle/working/'+name)
        print('KAGGLE: сохранено в /kaggle/working/'+name+' — скачайте из панели Output')
_save('/content/result.mp4', 'reels_v1_sharp.mp4')
print('ГОТОВО: result.mp4 скачан')


In [ ]:
# Ячейка 5 (опционально): Wav2Lip-синхронизация рта поверх + 50fps
import glob, os, urllib.request
if not os.path.isdir('w2l'):
    !git clone -q https://github.com/Rudrabha/Wav2Lip.git w2l
os.makedirs('w2l/checkpoints', exist_ok=True)
if not os.path.isfile('w2l/checkpoints/wav2lip.pth'):
    urllib.request.urlretrieve('https://github.com/Winfredy/SadTalker/releases/download/v0.0.2/wav2lip.pth','w2l/checkpoints/wav2lip.pth')
!sed -i 's/librosa.filters.mel(hp.sample_rate, hp.n_fft,/librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft,/' w2l/audio.py
!cd w2l && python inference.py --checkpoint_path checkpoints/wav2lip.pth --face /content/result.mp4 --audio ../examples/voice5.wav --outfile /content/sync.mp4 --pads 0 20 0 0
!ffmpeg -y -loglevel error -i /content/sync.mp4 -vf "minterpolate=fps=50:mi_mode=mci:mc_mode=aobmc:vsbmc=1" -c:v libx264 -crf 18 -pix_fmt yuv420p -movflags +faststart -c:a aac -b:a 128k /content/result_50fps.mp4
import google.colab.files as _gcf
getattr(_gcf,'download')('/content/result_50fps.mp4')
